In [ ]:
 !pip install fastapi uvicorn python-multipart langchain langchain-community langchain-ollama langchain-chroma pymupdf pydantic boto3 pyngrok langchain-experimental docx2txt unstructured python-pptx openpyxl msoffcrypto-tool rank_bm25 kiwipiepy langchain-huggingface sentence-transformers

In [12]:
import os

folders = ['api', 'core', 'schemas', 'services', 'data_storage', 'chroma_db']
for folder in folders:
    os.makedirs(folder, exist_ok=True)

In [ ]:
%%writefile schemas/dto.py
from typing import List, Optional
from pydantic import BaseModel

class ChatRequest(BaseModel):
    question: str
    space_id: Optional[str] = None

class SourceInfo(BaseModel):
    source: str
    page: Optional[int] = None


class ChatResponse(BaseModel):
    answer: str
    sources: List[SourceInfo]

class IngestRequest(BaseModel):
    space_id: str
    file_path: str


In [ ]:
%%writefile api/routes.py
from fastapi import APIRouter, Request
from fastapi import UploadFile, File, Form, BackgroundTasks
from schemas.dto import ChatRequest, ChatResponse, IngestRequest
from services.chatbot_service import process_chat, process_ingest

router = APIRouter()

@router.post("/chatbot", response_model=ChatResponse)
async def chat(req: ChatRequest, request: Request):
    return await process_chat(req, request.app)

@router.post("/ingest")
async def ingest_file(request: Request, background_tasks: BackgroundTasks, file: UploadFile = File(...), space_id: str = Form(...), user_id: str = Form("Unknown")):
    save_path = f"/content/data_storage/{file.filename}"
    with open(save_path, "wb") as buffer:
        content = await file.read()
        buffer.write(content)

    background_tasks.add_task(process_ingest, save_path, space_id, request.app, user_id)

    return {"status" : "processing", "message" : "파일 접수 완료. 백그라운드에서 학습 중입니다."}

In [ ]:
%%writefile core/config.py
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings  # 🌟 필수 임포트 추가

def ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)

DB_DIR = os.getenv("DB_DIR", "/content/chroma_db")
LOCAL_SOURCES = [p for p in os.getenv("LOCAL_SOURCES", "/content/data_storage").split(";") if p.strip()]
S3_BUCKET = os.getenv("S3_BUCKET", "").strip()
S3_PREFIXES = [p for p in os.getenv("S3_PREFIXES", "").split(";") if p.strip()]
S3_REGION = os.getenv("S3_REGION", "").strip()

INDEX_POLL_SECONDS = int(os.getenv("INDEX_POLL_SECONDS", "120"))
INDEX_STATE_PATH = os.getenv("INDEX_STATE_PATH", os.path.join(DB_DIR, "index_state.json"))

K_DENSE = int(os.getenv("K_DENSE", "20"))
FETCH_K = int(os.getenv("FETCH_K", "40"))
LAMBDA_MULT = float(os.getenv("LAMBDA_MULT", "0.65"))

ENABLE_BM25 = True
K_SPARSE = int(os.getenv("K_SPARSE", "20"))
K_FINAL = int(os.getenv("K_FINAL", "12"))
MAX_CONTEXT_CHARS = int(os.getenv("MAX_CONTEXT_CHARS", "9000"))
DEFAULT_SPACE_ID = os.getenv("DEFAULT_SPACE_ID", "default")

BASE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise and reliable business handover assistant.\n"
     "Your sole purpose is to help a successor understand and carry out their inherited responsibilities "
     "by answering questions strictly based on the [Context] provided below.\n\n"

     "════════════════════════════════════\n"
     "§ SOURCE OF TRUTH\n"
     "════════════════════════════════════\n"
     "- The ONLY valid source of information is the [Context] provided in each query.\n"
     "- Your internal knowledge, training data, or any external information MUST NOT be used — ever.\n"
     "- Even if you are highly confident about an answer from your internal knowledge, you MUST NOT use it.\n"
     "- Treat every question as if you have zero knowledge outside the given [Context].\n\n"

     "════════════════════════════════════\n"
     "§ ANSWER RULES\n"
     "════════════════════════════════════\n"
     "RULE 1 — FULL ANSWER:\n"
     "  If the [Context] contains sufficient information to fully answer the question,\n"
     "  provide a clear, structured, and helpful response based solely on that information.\n\n"

     "RULE 2 — PARTIAL INFORMATION:\n"
     "  If the [Context] contains only partial information relevant to the question,\n"
     "  answer only the parts that are directly supported by the [Context].\n"
     "  Explicitly state that the remaining information is not available in the provided materials.\n"
     "  DO NOT fill in gaps with assumptions or inferences.\n\n"

     "RULE 3 — NO INFORMATION:\n"
     "  If the [Context] contains NO information relevant to the question,\n"
     "  you MUST output EXACTLY and ONLY this keyword, with no additional text: 정보없음\n\n"

     "RULE 4 — NEVER DO THE FOLLOWING:\n"
     "  ✗ Do NOT guess, infer, assume, or extrapolate beyond the [Context].\n"
     "  ✗ Do NOT say 'According to the document', 'Based on the context', or reference file names.\n"
     "  ✗ Do NOT fabricate procedures, contacts, tools, systems, or any factual details.\n"
     "  ✗ Do NOT combine [Context] information with your internal knowledge.\n"
     "  ✗ Do NOT rephrase 정보없음 or add explanations when outputting it.\n\n"

     "RULE 5 — STRUCTURED COMPILATION (CRITICAL):\n"
     "  The [Context] may contain general procedures (manuals, checklists) and specific past events (incident logs, notes).\n"
     "  Do NOT ignore either. Instead, organize your response logically:\n"
     "  1. Outline the standard or general guidelines found in the context first.\n"
     "  2. Then, provide distinct sections for any specific past cases, historical incidents, or personal tips related to the question.\n"
     "  This ensures the successor receives both the official workflow and practical historical knowledge without them being mixed into a confusing answer.\n\n"

     "════════════════════════════════════\n"
     "§ LANGUAGE RULE\n"
     "════════════════════════════════════\n"
     "- Detect the language of the [Question] and respond in the same language.\n"
     "- If the [Question] is in Korean → respond in Korean.\n"
     "- If the [Question] is in English → respond in English.\n"
     "- For any other language → respond in that same language.\n"
     "- The only exception: if {answer_language} is explicitly specified and overrides this rule,\n"
     "  always follow {answer_language}.\n"
     "- The keyword 정보없음 is ALWAYS output as-is, regardless of language.\n\n"

     "════════════════════════════════════\n"
     "§ RESPONSE QUALITY GUIDELINES\n"
     "════════════════════════════════════\n"
     "- Be concise but complete. Avoid unnecessary filler or repetition.\n"
     "- Use bullet points or numbered steps when explaining procedures or multi-step processes.\n"
     "- Prioritize actionable, practical information that helps the successor do their job.\n\n"

     "COMPLETION RULES (CRITICAL):\n"
     "- You MUST always complete your response fully. Never stop mid-sentence or mid-list.\n"
     "- If you begin a numbered list or bullet list, you MUST include ALL items with their full content.\n"
     "- NEVER use '...' or any ellipsis to abbreviate or truncate content from the [Context].\n"
     "- NEVER summarize a list by omitting items. Every item must be written out in full.\n"
     "- If an answer contains steps, procedures, or enumerated items, ALL of them must appear in the response.\n"
    ),
    ("human",
     "[Context]\n{context}\n\n"
     "[Question]\n{question}\n\n"
     "[Answer]\n")
])

LLM_MODEL = os.getenv("LLM_MODEL", "gemma2:9b")
EMBED_MODEL = os.getenv("EMBED_MODEL", "jhgan/ko-sroberta-multitask")

llm = ChatOllama(
    model=LLM_MODEL,
    temperature=float(os.getenv("LLM_TEMPERATURE", "0.1")),
    num_predict=int(os.getenv("LLM_NUM_PREDICT", "2048")),
    timeout=int(os.getenv("LLM_TIMEOUT", "120")),
    base_url="http://127.0.0.1:11434"
)
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

def get_vectorstore() -> Chroma:
    return Chroma(
        persist_directory=DB_DIR,
        embedding_function=embedding_model,
        collection_metadata={"hnsw:space": "cosine"},
    )

In [ ]:
%%writefile services/document_service.py
import os, re, hashlib, unicodedata
from typing import List
from fastapi import FastAPI
from langchain_core.documents import Document
from langchain_community.document_loaders import (
    PyMuPDFLoader, TextLoader, Docx2txtLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata
from core.config import ENABLE_BM25, K_SPARSE, DEFAULT_SPACE_ID
from kiwipiepy import Kiwi
kiwi = Kiwi()

try:
    from langchain_community.retrievers import BM25Retriever
except Exception:
    BM25Retriever = None

def preprocess_text(text: str) -> str:
    if not text: return ""
    text = text.replace('\x00', '')
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def sanitize_metadata(meta: dict) -> dict:
    clean = {}
    for k, v in (meta or {}).items():
        if v is None: continue
        if isinstance(v, (str, int, float, bool)): clean[k] = v
        else: clean[k] = str(v)
    return clean

def custom_tokenize(text: str) -> List[str]:
    """Kiwi를 이용해 의미 있는 명사와 동사 어근만 추출하는 커스텀 토크나이저"""
    if not text:
        return []
    return [t.form for t in kiwi.tokenize(text) if t.tag.startswith('N') or t.tag.startswith('V')]

# def split_semantic_then_fallback(docs: List[Document]) -> List[Document]:
#     splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80, separators=["\n\n", "\n", " ", ""])
#     return splitter.split_documents(docs)
def split_semantic_then_fallback(docs: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=150,
        separators=["\n\n", "\n", ".", " "]
    )
    return splitter.split_documents(docs)

def stable_doc_id(source_uri: str, space_id: str) -> str:
    return hashlib.sha1(f"{space_id}::{source_uri}".encode("utf-8")).hexdigest()

def supported_ext(name: str) -> bool:
    nl = (name or "").lower()
    return nl.endswith((".pdf", ".txt", ".md", ".docx", ".doc"))

def load_and_chunk_from_path(path: str, source_label: str, space_id: str, doc_id: str) -> List[Document]:
    ext = os.path.splitext(path)[1].lower()
    docs = []
    try:
        if ext == ".pdf": docs = PyMuPDFLoader(path).load()
        elif ext in [".txt", ".md"]:
            raw = TextLoader(path, encoding="utf-8").load()[0].page_content
            docs = [Document(page_content=raw, metadata={"source": source_label})]
        elif ext in [".docx", ".doc"]: docs = Docx2txtLoader(path).load()
        else: return []
    except Exception as e:
        print(f"문서 파싱 에러 ({ext}): {e}")
        return []

    for d in docs:
        d.page_content = preprocess_text(d.page_content)
        d.metadata = d.metadata or {}
        d.metadata["source"] = source_label

    chunks = split_semantic_then_fallback(docs)
    cleaned = []
    MIN_CHUNK_LEN = 10

    for i, d in enumerate(chunks):
        if not d.page_content or not d.page_content.strip(): continue
        if len(d.page_content.strip()) < MIN_CHUNK_LEN: continue
        d.metadata = d.metadata or {}
        d.metadata.update({"doc_id": doc_id, "chunk_index": i, "source": source_label, "space_id": space_id})
        d.metadata = sanitize_metadata(d.metadata)
        cleaned.append(d)
    return filter_complex_metadata(cleaned)

# async def rebuild_bm25(app: FastAPI, space_id: str = DEFAULT_SPACE_ID) -> None:
#     if not ENABLE_BM25 or BM25Retriever is None:
#         app.state.bm25[space_id] = None
#         return
#     raw = app.state.vectordb._collection.get(where={"space_id": space_id}, include=["documents", "metadatas"])
#     docs = [Document(page_content=t, metadata=(m or {})) for t, m in zip(raw.get("documents", []), raw.get("metadatas", [])) if t and t.strip()]
#     if docs:
#         bm25 = BM25Retriever.from_documents(docs)
#         bm25.k = K_SPARSE
#         app.state.bm25[space_id] = bm25
#     else:
#         app.state.bm25[space_id] = None
async def rebuild_bm25(app: FastAPI, space_id: str = DEFAULT_SPACE_ID) -> None:
    if not ENABLE_BM25 or BM25Retriever is None:
        app.state.bm25[space_id] = None
        return
    raw = app.state.vectordb._collection.get(where={"space_id": space_id}, include=["documents", "metadatas"])
    docs = [Document(page_content=t, metadata=(m or {})) for t, m in zip(raw.get("documents", []), raw.get("metadatas", [])) if t and t.strip()]
    if docs:
        bm25 = BM25Retriever.from_documents(docs, preprocess_func=custom_tokenize)
        bm25.k = K_SPARSE
        app.state.bm25[space_id] = bm25
    else:
        app.state.bm25[space_id] = None

In [ ]:
%%writefile services/chatbot_service.py
import time, math, os, asyncio, traceback
from datetime import datetime
from typing import List
from fastapi import FastAPI, HTTPException
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from core.config import *
from schemas.dto import ChatRequest, ChatResponse, SourceInfo
from services.document_service import load_and_chunk_from_path, stable_doc_id, rebuild_bm25
from sentence_transformers import CrossEncoder
reranker_model = CrossEncoder("Dongjin-kr/ko-reranker", max_length=512, device="cuda")

def is_korean(text: str) -> bool:
    return any(("\uAC00" <= c <= "\uD7A3") or ("\u1100" <= c <= "\u11FF") or ("\u3130" <= c <= "\u318F") for c in (text or ""))

def build_context(docs: List[Document], max_chars: int = MAX_CONTEXT_CHARS) -> str:
    parts, total = [], 0
    for d in docs:
        text = (d.page_content or "").strip()
        if not text: continue

        source_name = (d.metadata or {}).get("source", "알 수 없는 문서")
        labeled_text = f"[문서명: {source_name}]\n{text}"

        if total + len(labeled_text) > max_chars:
            break
        parts.append(labeled_text)
        total += len(labeled_text)
    return "\n\n---\n\n".join(parts)

def cosine(a: List[float], b: List[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na, nb = math.sqrt(sum(x * x for x in a)), math.sqrt(sum(y * y for y in b))
    return dot / (na * nb + 1e-12)

def rerank_with_cross_encoder(query: str, docs: List[Document], top_k: int) -> List[Document]:
    if not docs: return []

    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker_model.predict(pairs)

    scored_docs = list(zip(scores, docs))
    scored_docs.sort(key=lambda x: x[0], reverse=True)

    return [doc for score, doc in scored_docs[:top_k] if score > 0.0]

def reciprocal_rank_fusion(dense_docs: List[Document], sparse_docs: List[Document], k: int = 60) -> List[Document]:
    rrf_scores = {}
    doc_map = {}

    for rank, doc in enumerate(dense_docs):
        doc_key = str(doc.metadata.get("doc_id", "")) + ":" + str(doc.metadata.get("chunk_index", ""))
        rrf_scores[doc_key] = rrf_scores.get(doc_key, 0.0) + 1.0 / (rank + k)
        doc_map[doc_key] = doc

    if sparse_docs:
        for rank, doc in enumerate(sparse_docs):
            doc_key = str(doc.metadata.get("doc_id", "")) + ":" + str(doc.metadata.get("chunk_index", ""))
            rrf_scores[doc_key] = rrf_scores.get(doc_key, 0.0) + 1.0 / (rank + k)
            doc_map[doc_key] = doc

    sorted_keys = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return [doc_map[key] for key in sorted_keys]

async def process_chat(req: ChatRequest, app: FastAPI) -> ChatResponse:
    start = time.time()
    q = (req.question or "").strip()
    if not q: raise HTTPException(status_code=400, detail="question is required")

    space_id = (req.space_id or DEFAULT_SPACE_ID).strip() or DEFAULT_SPACE_ID
    vectordb = app.state.vectordb
    bm25 = getattr(app.state, "bm25", {}).get(space_id)
    answer_language = "Korean" if is_korean(q) else "English"
    search_query = q

    retriever = vectordb.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": K_DENSE,
            "fetch_k": FETCH_K,
            "lambda_mult": LAMBDA_MULT,
            "filter": {"space_id": space_id}
        }
    )
    dense_candidates = list(await asyncio.to_thread(retriever.invoke, search_query))

    sparse_candidates = []
    if ENABLE_BM25 and bm25 is not None:
        try:
            bm25.k = K_SPARSE
            raw_sparse = await asyncio.to_thread(bm25.invoke, search_query)
            sparse_candidates = [d for d in raw_sparse if (d.metadata or {}).get("space_id") == space_id]
        except Exception as e:
            print(f"BM25 Search Error: {e}")

    rrf_candidates = reciprocal_rank_fusion(dense_candidates, sparse_candidates, k=60)
    top_candidates = rrf_candidates[:15]
    final_docs = await asyncio.to_thread(rerank_with_cross_encoder, q, top_candidates, K_FINAL)

    if not final_docs:
        answer = "관련 자료에서 답을 찾지 못했습니다." if answer_language == "Korean" else "I don't have information about that."
        return ChatResponse(answer=answer, sources=[])

    context = build_context(final_docs, max_chars=MAX_CONTEXT_CHARS)
    chain = BASE_PROMPT | llm | StrOutputParser()
    answer = (await chain.ainvoke({"context": context, "question": q, "answer_language": answer_language}) or "").strip()

    sources = []

    if "정보없음" in answer.replace(" ", ""):
        return ChatResponse(
            answer="관련 자료에서 답을 찾지 못했습니다. 다른 키워드로 질문해 주시거나 추가 자료를 업로드해 주세요.",
            sources=[]
        )

    no_answer_keywords = [
        "관련 자료에서 답을 찾지 못했",
        "정보를 가지고 있지 않습니다",
        "정보가 없습니다",
        "정보는 없습니다",
        "찾을 수 없습니다",
        "찾지 못했습니다",
        "알 수 없습니다",
        "언급되어 있지 않습니다",
        "내용이 없습니다",
        "답변할 수 없습니다",
        "제공되지 않았습니다",
        "I don't have information about",
        "not mentioned in the",
        "not available in the provided",
    ]

    if not any(keyword in answer for keyword in no_answer_keywords):
        seen_sources = set()
        for d in final_docs:
            meta = d.metadata or {}
            source_name = meta.get("source", "unknown")
            page_num = meta.get("page")
            page_val = int(page_num) + 1 if page_num is not None else None

            uniq_key = f"{source_name}_{page_val}"
            if uniq_key not in seen_sources:
                sources.append(SourceInfo(source=source_name, page=page_val))
                seen_sources.add(uniq_key)

    print(f"⏱️ 챗봇 응답 소요 시간: {time.time() - start:.2f}초")
    return ChatResponse(answer=answer, sources=sources)

async def process_ingest(file_path: str, space_id: str, app: FastAPI, user_id: str = "Unknown") -> dict:
    try:
        if not os.path.exists(file_path):
            print(f"[ERROR] File not found: {file_path}", flush=True)
            return {"status": "error", "message": f"File not found: {file_path}"}

        vectordb = app.state.vectordb
        uri, source_label = f"file://{os.path.abspath(file_path)}", os.path.basename(file_path)
        doc_id = stable_doc_id(uri, space_id)

        chunks = await asyncio.to_thread(load_and_chunk_from_path, file_path, source_label, space_id, doc_id)

        if not chunks: return {"status": "skipped", "message": "지원하지 않는 확장자이거나 추출할 텍스트가 없습니다."}

        async with app.state.write_lock:
            if old_ids := vectordb._collection.get(where={"doc_id": doc_id}).get("ids", []): vectordb._collection.delete(ids=old_ids)
            batch_size = 5
            skipped = 0
            for i in range(0, len(chunks), batch_size):
                batch_chunks = chunks[i : i + batch_size]

                try:
                    await asyncio.to_thread(vectordb.add_documents, batch_chunks)
                except Exception:
                    for single_chunk in batch_chunks:
                        try:
                            await asyncio.to_thread(vectordb.add_documents, [single_chunk])
                        except Exception as chunk_err:
                            skipped += 1
                            print(f"⚠️ [SKIP] 청크 임베딩 실패 (건너뜀): {chunk_err}", flush=True)

                now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                current_chunk = min(i + batch_size, len(chunks))
                print(f"[{now}] [User: {user_id} | Space: {space_id}] 임베딩 진행 중 ...({current_chunk}/{len(chunks)})", flush=True)
                await asyncio.sleep(0.1)

            if skipped:
                print(f"⚠️ 총 {skipped}개 청크 임베딩 실패로 제외됨", flush=True)

        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{now}] [User: {user_id} | Space: {space_id}] 임베딩 완료", flush=True)
        async with app.state.rebuild_lock: await rebuild_bm25(app, space_id=space_id)
        return {"status": "success", "message": f"성공적으로 {len(chunks)}개의 청크를 DB에 추가했습니다.", "space_id": space_id}
    except Exception as e:
        print(f"[ERROR] 백그라운드 에러 발생")
        print(f"[ERROR] 에러 원인: {str(e)}", flush=True)
        traceback.print_exc()
        return {"status": "error", "message": str(e)}

In [ ]:
%%writefile main.py
import asyncio
from contextlib import asynccontextmanager
from fastapi import FastAPI
from core.config import DB_DIR, ensure_dir, get_vectorstore
from api.routes import router

@asynccontextmanager
async def lifespan(app: FastAPI):
    ensure_dir(DB_DIR)

    app.state.vectordb = get_vectorstore()
    app.state.bm25 = {}
    app.state.write_lock = asyncio.Lock()
    app.state.rebuild_lock = asyncio.Lock()
    app.state.stop_event = asyncio.Event()

    print(">> Server Started (Layered Architecture)")
    yield

    app.state.stop_event.set()
    print(">> Server Shutdown")

app = FastAPI(title="RAG API (Layered)", lifespan=lifespan)

app.include_router(router)

In [ ]:
!apt-get update
!apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

subprocess.Popen(['ollama', 'serve'])
print("서버 켜지는 중... 10초 대기")
time.sleep(10)

!ollama pull gemma2:9b

In [ ]:
import os
import nest_asyncio
import uvicorn
import subprocess
import time
from pyngrok import ngrok, conf
from pyngrok.exception import PyngrokNgrokError
from main import app

os.system("pkill -9 ngrok")
time.sleep(1)

nest_asyncio.apply()

NGROK_TOKEN = "3AW7vlF3pmyC2QQrhfmmvLc2IB8_2399FN1LYiPPybrHscyQx"
conf.get_default().auth_token = NGROK_TOKEN

try:
    public_url = ngrok.connect(8000)
    print("=" * 50)
    print(f"🌍🌍🌍🌍🌍\n 완성! 외부 접속 주소: {public_url}\n🌍🌍🌍🌍🌍")
    print("=" * 50)

    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve()

except PyngrokNgrokError as e:
    print("\n🚨 ngrok이 켜지다 죽었습니다! 아래 상세 로그를 확인해주세요:")
    for log in e.ngrok_logs:
        print(log)
    print("\n에러 메시지:", e)